# CYMEK V5.1 Canary-v2 — pinned T4/CUDA operator launcher

This notebook executes the **pre-registered V5.1 Canary-v2 formation qualification** against the exact frozen scientific commit below. It is deliberately fail-closed: it verifies critical Git blobs, runs the static and CPU qualification gates, isolates CUDA preflight from scientific checkpoint state, persists training to Google Drive, resumes compatible state instead of restarting, and consumes the sealed split only through the one-shot finalizer.

**Frozen scientific executable:** `4470a34b7e2d4b2d673c328ef962c84d8e075b89`

Run cells in order. Re-running after a Colab disconnect is safe when the Drive root contains the same executable binding. Never delete `state/`, `receipts/`, or `SEALED_CONSUMPTION.json` to force a rerun.


In [ ]:
import sys, os, json, hashlib, subprocess, time
from pathlib import Path

REPO = Path("/content/An-Ra-the-new-AGI-v51-canary-v2")
REMOTE = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
BRANCH = "cymek-v51-canary-v2"
SCIENCE = "4470a34b7e2d4b2d673c328ef962c84d8e075b89"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", "--depth", "200", REMOTE, str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REMOTE], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH, "--depth", "200"], check=True)

subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "-q", SCIENCE], check=True)
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert HEAD == SCIENCE, (HEAD, SCIENCE)

EXPECTED_BLOBS = {
    "anra_v5/v51_canary_v2_run.py": "5457b7fe60eb253013584437987edc9148a2de46",
    "anra_v5/v51_canary_run.py": "87845ec0795fac16d496ee39119a385cb9f223e7",
    "anra_v5/v51_canary_data.py": "56d25296cc706e5c0d8372a7c30ba94942d6c0f5",
    "experiments/V5_1_CANARY_V2/PREREGISTRATION.json": "fcfcb157ce2d4bdafeffe39ef604668546380023",
    "tools/validate_v51_canary_v2.py": "59061560e9e30a8c5a2cd6aab77e793990cd5fd8",
    "tests/test_v51_canary.py": "63a4005ed0bc80ac99c95a1972605fc35e05b503",
    "tests/test_v51_canary_v2.py": "ac48fdd12640be1298d3ba639553f17e7aa73a46",
    "tests/test_v51_canary_v2_durability.py": "3ad3e3cd4259cc25af1f683b677f0f3e2c482950",
    "v5_model/core.py": "7cf64b6f557a0556c074e5f61adfc86702f4c725",
    "v5_training/production_backend.py": "72646373c1890e19deaeef634b3f4a0dbdf99632",
    "v5_training/checkpoint.py": "6bd1d04dad43e402b9acff9128d1ad56528a37d0",
    "v5_training/step.py": "bf1d0411be249d2e6e4f8634381124e4f6c7e92f",
}
for rel, expected in EXPECTED_BLOBS.items():
    got = subprocess.check_output(["git", "-C", str(REPO), "hash-object", rel], text=True).strip()
    assert got == expected, f"FAIL_CLOSED BLOB: {rel}: {got} != {expected}"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "numpy"], check=True)
subprocess.run([sys.executable, "tools/validate_v51_canary_v2.py"], cwd=REPO, check=True)

CPU_TESTS = [
    "tests/test_v51_canary.py",
    "tests/test_v51_canary_v2.py",
    "tests/test_v51_canary_v2_durability.py",
    "tests/test_v5_production_backend.py",
    "tests/test_v5_checkpoint_adapter.py",
]
subprocess.run([sys.executable, "-m", "pytest", *CPU_TESTS, "-q"], cwd=REPO, check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. In Colab choose Runtime -> Change runtime type -> T4 GPU.")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)

PREREG_PATH = REPO / "experiments/V5_1_CANARY_V2/PREREGISTRATION.json"
PREREG_RAW = PREREG_PATH.read_bytes()
PREREG = json.loads(PREREG_RAW)
assert PREREG["schema"] == "anra-v51-canary-v2-preregistration/v1"
assert PREREG["model"]["parameter_count"] == 10_227_456
assert PREREG["training"]["target_updates"] == 360
assert PREREG["training"]["token_budget"] == 1_474_560
assert PREREG["model"]["output_path"] == "tied full softmax, canonical only"

print("QUALIFICATION: PASS")
print("SCIENCE:", SCIENCE)
print("GPU:", GPU_NAME, "| VRAM GiB:", GPU_VRAM_GIB)
print("MODEL PARAMS:", f"{PREREG['model']['parameter_count']:,}")
print("ENDPOINT:", PREREG["training"]["target_updates"], "updates /", f"{PREREG['training']['token_budget']:,}", "tokens")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = Path("/content/drive/MyDrive/CYMEK/V5_1_CANARY_V2")
OUT.mkdir(parents=True, exist_ok=True)
RECEIPTS = OUT / "receipts"
STATE = OUT / "state"
SEALED_LOCK = OUT / "SEALED_CONSUMPTION.json"
FINAL = RECEIPTS / "FINALIZATION.json"
BINDING = OUT / "EXECUTABLE_BINDING.json"
FROZEN_PREREG = OUT / "PREREGISTRATION_FROZEN.json"

ENV = dict(os.environ)
ENV["V51_CANARY_V2_ROOT"] = str(OUT)
ENV["PYTHONUNBUFFERED"] = "1"

binding = {
    "schema": "anra-v51-canary-v2-operator-binding/v1",
    "scientific_executable_commit": SCIENCE,
    "runner": "anra_v5/v51_canary_v2_run.py",
    "preregistration_raw_sha256": hashlib.sha256(PREREG_RAW).hexdigest(),
    "critical_git_blobs": EXPECTED_BLOBS,
    "drive_root": str(OUT),
    "claim_ceiling": PREREG["claim_ceiling"],
}
if BINDING.exists():
    existing = json.loads(BINDING.read_text(encoding="utf-8"))
    assert existing == binding, "FAIL_CLOSED: existing Drive executable binding differs"
else:
    BINDING.write_text(json.dumps(binding, indent=2, sort_keys=True) + "\n", encoding="utf-8")

if FROZEN_PREREG.exists():
    assert FROZEN_PREREG.read_bytes() == PREREG_RAW, "FAIL_CLOSED: Drive preregistration bytes differ"
else:
    FROZEN_PREREG.write_bytes(PREREG_RAW)

def run_mode(mode, *, cuda=False, capture=False, check=True):
    cmd = [sys.executable, "-m", "anra_v5.v51_canary_v2_run", "--mode", mode]
    if cuda:
        cmd.append("--cuda")
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(
        cmd, cwd=REPO, env=ENV,
        text=True, capture_output=capture, check=check,
    )

def scan():
    proc = run_mode("scan", capture=True)
    print(proc.stdout)
    payload = json.loads(proc.stdout)
    assert payload["lineage"] == "v51-canary-v2"
    assert payload["target_updates"] == 360
    return payload

data_receipt = RECEIPTS / "DATA.json"
required_split_files = [OUT / "training.jsonl", OUT / "development.jsonl", OUT / "sealed.jsonl"]
if data_receipt.exists():
    data = json.loads(data_receipt.read_text(encoding="utf-8"))
    assert data["schema"] == "anra-v51-canary-v2-data-receipt/v1"
    assert int(data["v2_fresh_seed"]) == int(PREREG["seed"])
    assert data["contamination_screen"]["clean"] is True
    assert all(p.is_file() for p in required_split_files), "FAIL_CLOSED: DATA receipt exists but split file is missing"
    print("DATA: existing bound V2 dataset preserved")
else:
    run_mode("prepare")
    data = json.loads(data_receipt.read_text(encoding="utf-8"))
    assert data["contamination_screen"]["clean"] is True

# This preflight uses a temporary checkpoint lineage inside the runner and
# writes only the final preflight receipt into the scientific root.
run_mode("preflight", cuda=True)
pre = json.loads((RECEIPTS / "PREFLIGHT.json").read_text(encoding="utf-8"))
assert pre["schema"] == "anra-v51-canary-v2-preflight/v1"
assert pre["scientific_state_untouched"] is True
assert pre["isolated_lineage"] == "v51-canary-v2-preflight"

SCAN = scan()
assert SCAN["action"] in {"START", "RESUME", "COMPLETE"}, SCAN
print("SAFE ACTION:", SCAN["action"])


In [ ]:
TRAINING = RECEIPTS / "TRAINING.json"

def durable_progress():
    if not TRAINING.exists():
        return {"update": 0, "status": "NO_TRACE", "loss": None, "epoch": None}
    try:
        body = json.loads(TRAINING.read_text(encoding="utf-8"))
        trace = list(body.get("trace", []))
        if not trace:
            return {"update": 0, "status": body.get("status"), "loss": None, "epoch": None}
        last = trace[-1]
        return {
            "update": int(last["update"]),
            "status": body.get("status"),
            "loss": last.get("loss"),
            "epoch": last.get("epoch"),
        }
    except Exception as exc:
        return {"update": None, "status": f"READ_ERROR:{type(exc).__name__}", "loss": None, "epoch": None}

if SCAN["action"] != "COMPLETE":
    cmd = [
        sys.executable, "-m", "anra_v5.v51_canary_v2_run",
        "--mode", "run", "--cuda",
        "--updates", "360", "--checkpoint-every", "24",
    ]
    print("STARTING/RESUMING FIXED V2 ENDPOINT")
    print("$", " ".join(cmd), flush=True)
    started = time.time()
    proc = subprocess.Popen(cmd, cwd=REPO, env=ENV)
    try:
        while proc.poll() is None:
            time.sleep(30)
            p = durable_progress()
            elapsed = int(time.time() - started)
            update = p["update"]
            pct = None if update is None else round(100.0 * update / 360.0, 1)
            print(
                f"[{elapsed:>5}s] durable={update}/360 ({pct}%) "
                f"epoch={p['epoch']} loss={p['loss']} trace={p['status']}",
                flush=True,
            )
    except KeyboardInterrupt:
        print("Notebook interrupted. Terminating child; last published checkpoint/trace on Drive remains resumable.")
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
        raise
    rc = proc.wait()
    print("TRAIN RETURN CODE:", rc)
    if rc != 0:
        raise RuntimeError("Canary-v2 training stopped/failed. Preserve Drive state and inspect the emitted receipt/checkpoint before retry.")

training = json.loads(TRAINING.read_text(encoding="utf-8"))
trace = list(training.get("trace", []))
assert training["schema"] == "anra-v51-canary-v2-training/v1"
assert training["status"] == "COMPLETE_ENDPOINT"
assert [int(r["update"]) for r in trace] == list(range(1, 361))
assert int(trace[-1]["tokens_seen"]) == 1_474_560
assert training["all_lr_match"] is True

POST_TRAIN_SCAN = scan()
assert POST_TRAIN_SCAN["action"] in {"RESUME", "COMPLETE"}, POST_TRAIN_SCAN
print("TRAINING ENDPOINT: PASS | durable trace rows:", len(trace))


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if FINAL.exists():
    final = json.loads(FINAL.read_text(encoding="utf-8"))
    print("FINALIZATION already exists; sealed split will NOT be consumed again.")
else:
    if SEALED_LOCK.exists():
        marker = json.loads(SEALED_LOCK.read_text(encoding="utf-8"))
        if marker.get("status") != "CONSUMED_AND_FINALIZED":
            raise RuntimeError(
                "FAIL_CLOSED: sealed-consumption marker exists without completed finalization. "
                "Do not rerun sealed evaluation; follow the amendment/regeneration path."
            )

    # Development evaluation is diagnostic only. The notebook does not branch
    # on its result: finalization follows automatically, preventing outcome-based stopping.
    run_mode("evaluate", cuda=True)
    dev_receipt = json.loads((RECEIPTS / "EVALUATION.json").read_text(encoding="utf-8"))
    assert dev_receipt["global_update"] == 360

    run_mode("finalize", cuda=True)
    assert FINAL.exists(), "FAIL_CLOSED: finalizer returned without FINALIZATION.json"
    final = json.loads(FINAL.read_text(encoding="utf-8"))

marker = json.loads(SEALED_LOCK.read_text(encoding="utf-8"))
assert marker["status"] == "CONSUMED_AND_FINALIZED"
assert final["executable_commit"] == SCIENCE
assert int(final["global_update"]) == 360
assert int(final["cumulative_tokens"]) == 1_474_560
assert final["verdict"] in {
    "CANARY_V2_PASS",
    "CANARY_V2_FAIL_FORMATION",
    "CANARY_V2_FAIL_ENGINEERING",
}
assert all(final["mechanical_gates"].values()) or final["verdict"] == "CANARY_V2_FAIL_ENGINEERING"

operator_result = {
    "schema": "anra-v51-canary-v2-operator-result/v1",
    "scientific_executable_commit": SCIENCE,
    "gpu": GPU_NAME,
    "gpu_vram_gib": GPU_VRAM_GIB,
    "verdict": final["verdict"],
    "global_update": final["global_update"],
    "cumulative_tokens": final["cumulative_tokens"],
    "checkpoint_sha256": final["checkpoint_sha256"],
    "development_overall_exact_with_valid_eos": final["development_overall_exact_with_valid_eos"],
    "sealed_overall_exact_with_valid_eos": final["sealed_overall_exact_with_valid_eos"],
    "formation_gates": final["formation_gates"],
    "mechanical_gates": final["mechanical_gates"],
    "finalization_receipt_file_sha256": sha256_file(FINAL),
    "sealed_marker_file_sha256": sha256_file(SEALED_LOCK),
    "claim_ceiling": final["claim_ceiling"],
}
(OUT / "OPERATOR_RESULT.json").write_text(
    json.dumps(operator_result, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("\n========== V5.1 CANARY-v2 RESULT ==========")
print("VERDICT:", final["verdict"])
print("DEV exact+EOS:", final["development_overall_exact_with_valid_eos"])
print("SEALED exact+EOS:", final["sealed_overall_exact_with_valid_eos"])
print("FORMATION GATES:", json.dumps(final["formation_gates"], indent=2))
print("MECHANICAL GATES:", json.dumps(final["mechanical_gates"], indent=2))
print("NEXT IF PASS:", final["next_if_pass"])
print("NEXT IF FAIL:", final["next_if_fail"])


In [ ]:
import zipfile

bundle = OUT / "CYMEK_V51_CANARY_V2_RESULTS.zip"
include = [
    BINDING,
    FROZEN_PREREG,
    SEALED_LOCK,
    OUT / "OPERATOR_RESULT.json",
]
include += sorted(p for p in RECEIPTS.glob("*.json") if p.is_file())

tmp_bundle = Path("/content/CYMEK_V51_CANARY_V2_RESULTS.zip")
if tmp_bundle.exists():
    tmp_bundle.unlink()

with zipfile.ZipFile(tmp_bundle, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    seen = set()
    for p in include:
        if not p.is_file():
            continue
        arc = str(p.relative_to(OUT))
        if arc in seen:
            continue
        seen.add(arc)
        zf.write(p, arcname=arc)

bundle.write_bytes(tmp_bundle.read_bytes())
bundle_sha = sha256_file(bundle)
manifest = {
    "schema": "anra-v51-canary-v2-result-bundle/v1",
    "bundle": bundle.name,
    "sha256": bundle_sha,
    "included_files": sorted(seen),
    "checkpoint_state_excluded": True,
    "raw_train_dev_sealed_rows_excluded": True,
}
(OUT / "RESULT_BUNDLE_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("RESULT BUNDLE:", bundle)
print("SHA256:", bundle_sha)
print("Checkpoint state remains separately in Drive:", STATE)

from google.colab import files
files.download(str(bundle))
